Pairwise Alpha

In [3]:
import csv
from itertools import combinations
from nltk.metrics.agreement import AnnotationTask
import statistics
import pandas as pd
import json
import numpy as np
from scipy.stats import t


pair_sample_number = []


def load_all_annotations_of_all_annotators(all_annotations, current_data):
     
     # let the annotations_asr only left with the existing extract_id of data_ccc, for only 2090 extracts
     annotations_asr = all_annotations[all_annotations['extract_id'].isin(current_data['extract_id'])]

     # remove the extracts that has only one annotation
     annotations_asr = annotations_asr[annotations_asr['extract_id'].map(annotations_asr['extract_id'].value_counts()) > 1]
     
     return annotations_asr
     

def get_grouped_annotators(group_annotators, annotations_asr):
    # remove the annotators in batches_annotators.json that are not in annotations_asr
    for group in group_annotators:
        group_annotators[group] = [antr for antr in group_annotators[group] if antr in annotations_asr['anonymised_participant_id'].values]

    # remove the batches with less than 2 annotators
    group_annotators = {group: group_annotators[group] for group in group_annotators if len(group_annotators[group]) >= 2}

    # check if there are any batches with less than 2 annotators, print out the names of the batches and the annotators in them
    for group in group_annotators:
        if len(group_annotators[group]) < 2:
            print(group, group_annotators[group])
            print('There are some batches with less than 2 annotators')

    return group_annotators


# if calculating the agreement among LLM and human
def add_llm_scores(current_data, annotations_asr, group_annotators):
    #concatenate data_ccc after annotations_asr
    ccc_renamed = current_data.copy()
    del ccc_renamed['score']
    ccc_renamed = ccc_renamed.rename(columns={'output_score': 'score'})
    ccc_renamed['anonymised_participant_id'] = 'llm'
    annotations_asr = pd.concat([annotations_asr, ccc_renamed], join='inner')

    # add 'llm' to each annotator batch
    for group in group_annotators:
        group_annotators[group].append('llm')

    return annotations_asr, group_annotators


def create_pairwise_alpha_csv(group_annotators, triples_results):
    # calculating Krippendorff's alpha for every pair of annotators in a group
    with open("./pairwise_alpha.csv", 'w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['batch', 'annotator_1', 'annotator_2', 'alpha']) #header
        
        for i, group in enumerate(group_annotators): # iterating over annotators' groups
            pairs_in_group = list(combinations(group_annotators[group], 2)) # pairs combinations in every group
            for pair in pairs_in_group: # iterating over every pair of annotators in a group
                responses_list = [] # all the responses for 2 annotators in a pair
                for triple in triples_results: # iterating over the responses
                    
                    if pair[0] == triple[0]: # matching annotators' IDs
                        responses_list.append((triple[0],triple[1],triple[2])) # putting all the responses of annotator_1  in a tuple

                    if pair[1] == triple[0]:
                        responses_list.append((triple[0],triple[1],triple[2])) # putting all the responses of annotator_2  in a tuple

                # get the set of item_ids for each annotator in the pair
                ann1 = {item_id for (a, item_id, label) in responses_list if a == pair[0]}
                ann2 = {item_id for (a, item_id, label) in responses_list if a == pair[1]}

                # find the common item_ids between the two annotators
                common_items = ann1 & ann2
                if pair[0] == 'llm' or pair[1] == 'llm':
                    pair_sample_number.append(len(common_items))
                    # print(f"Group {group}, Pair {pair}: Number of common items = {len(common_items)}")

                # filter responses_list to only include responses for the common item_ids
                filtered_responses = [
                    (a, item_id, label)
                    for (a, item_id, label) in responses_list
                    if item_id in common_items
                ]

                num_items = len(common_items)
                num_responses = len(filtered_responses)

                # only calculate alpha if there are at least 2 common items, otherwise it will be zero division error
                if num_items > 1:
                    try:
                        t = AnnotationTask(data=filtered_responses)
                        k_alpha = round(t.alpha(), 3)
                    except ZeroDivisionError:
                        k_alpha = 'zero division'
                else:
                    k_alpha = 'insufficient data'

                writer.writerow([group,pair[0],pair[1],k_alpha])


def calculate_mean_alpha_per_annotator(annotations_asr):  # mean alpha per annotator (comparing with experts, crowd, or both)
    # reading 'pairwise_alpha.csv'
    pairwise_agreement = pd.read_csv("./pairwise_alpha.csv")
    pairs_in_tuples = [] # list with tuples (pairs of annotators) and alpha per pair

    # making a list of unique annotators
    list_of_unique_annotators = list(annotations_asr.groupby('anonymised_participant_id').groups.keys())

    for i,row in pairwise_agreement.iterrows():
        pairs_in_tuples.append([(row['annotator_1'],row['annotator_2']),row['alpha']])
    ann_mean = [] # list of mean alpha per annotator
    for unique in list_of_unique_annotators: # iterating over the list of unique annotators
        # print(unique)

        ann_values_with_crowd = []
        ann_values_with_expert = []
        ann_values_with_llm = []

        for pair in pairs_in_tuples:
            if unique in pair[0]: # matching annotators IDs as the first element of a pair
                # print(pair)
                if 'llm' in unique: # unique == llm
                    if 'unknown' in pair[0][0] or 'unknown' in pair[0][1]:
                        ann_values_with_expert.append(float(pair[1]))
                    else:
                        ann_values_with_crowd.append(float(pair[1]))
                elif 'unknown' in unique: # unique == expert
                    if 'llm' in pair[0]:
                        ann_values_with_llm.append(float(pair[1]))
                    else:
                        ann_values_with_expert.append(float(pair[1]))
                else: # unique == crowd
                    if 'llm' in pair[0]:
                        ann_values_with_llm.append(float(pair[1]))
                    else:
                        ann_values_with_crowd.append(float(pair[1]))
        
        mean_kappa = []
        moe_mean = []
        for values in [ann_values_with_llm, ann_values_with_expert, ann_values_with_crowd]: # there is an order!
            # print(values)
            if values != []: # there's one batch with only 1 annotator, so there's no pair
                mean_kappa.append(round(statistics.mean(values),3))
                moe_mean.append(round(agreement_moe(values),3))
            else:
                mean_kappa.append('no pair')
                moe_mean.append('no pair')
        all_pair_scores = ann_values_with_crowd + ann_values_with_expert + ann_values_with_llm
        mean_kappa.append(round(statistics.mean(all_pair_scores),3)) if all_pair_scores != [] else mean_kappa.append('no pair')
        moe_mean.append(round(agreement_moe(all_pair_scores),3)) if all_pair_scores != [] else moe_mean.append('no pair')
        
        ann_mean.append([unique,mean_kappa, moe_mean])

    mean_alpha = pd.DataFrame(ann_mean,columns=['anonymised_participant_id','mean_alpha', 'moe'])

    return mean_alpha


# calculating the margin of error for the agreement scores
def agreement_moe(iaa_values_of_each_pair, confidence=0.95):
    iaa_values = np.array(iaa_values_of_each_pair)
    k = len(iaa_values)              
    s = np.std(iaa_values, ddof=1)  
    se = s / np.sqrt(k)             
    df = k - 1                     
    t_value = t.ppf(1 - (1 - confidence) / 2, df)
    moe = t_value * se
    return moe


def calculate_pairwise_agreement(data_name, data_type="data_path"):
    if data_type == "data_path":
        with open(data_name, 'r') as f:
            data_content = pd.read_csv(f)
    elif data_type == "data_df":
        data_content = data_name

    with open("./ready_annotations.csv", 'r') as f:
        annotations_asr_original = pd.read_csv(f)

    with open("./batches_annotators.json", 'r') as infile:
        annotators_original = json.load(infile)

    annotations_asr = load_all_annotations_of_all_annotators(annotations_asr_original, data_content)
    group_annotators = get_grouped_annotators(annotators_original, annotations_asr)
    annotations_asr, group_annotators = add_llm_scores(data_content, annotations_asr, group_annotators)
    # converting df to a list with all responses
    triples_results = annotations_asr.values.tolist()
    create_pairwise_alpha_csv(group_annotators, triples_results)
    mean_alpha = calculate_mean_alpha_per_annotator(annotations_asr)
    return mean_alpha

In [52]:
# for Table 1: IAA scores for LLMs and their margin of error for all samples

import pandas as pd
import os
from sklearn.metrics import confusion_matrix

# load the LLM output data
input_dir = "./gpt and llama"

results = []

for filename in os.listdir(input_dir):
    if filename.endswith(".csv"):
        model_name = os.path.splitext(filename)[0]
        if model_name != 'model_confusion_matrix_summary':
            filepath = os.path.join(input_dir, filename)
            alphas = calculate_pairwise_agreement(filepath)
            # get the alpha for 'llm'
            alpha_llm = alphas[alphas['anonymised_participant_id'] == 'llm']['mean_alpha'].values[0]
            alpha_moe = alphas[alphas['anonymised_participant_id'] == 'llm']['moe'].values[0]
        
            results.append((model_name, alpha_llm, alpha_moe))

results # the alpha score list for LLMs and their margin of error are in the order of [llm vs llm, llm vs expert, llm vs crowd, llm vs all (expert + crowd)]
    

/Users/yahui/Contentious-term-recognition-with-LLM/contentious/lib/python3.9/site-packages/numpy/_core/_methods.py:218: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/yahui/Contentious-term-recognition-with-LLM/contentious/lib/python3.9/site-packages/numpy/_core/_methods.py:210: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/Users/yahui/Contentious-term-recognition-with-LLM/contentious/lib/python3.9/site-packages/numpy/_core/_methods.py:218: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/yahui/Contentious-term-recognition-with-LLM/contentious/lib/python3.9/site-packages/numpy/_core/_methods.py:210: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/Users/yahui/Contentious-term-recognition-with-LLM/contentious/lib/python3.9/site-packages/numpy/_core/_meth

[('gpt-4o  ori_en_CandR',
  ['no pair', 0.653, 0.524, 0.531],
  ['no pair', np.float64(0.058), np.float64(0.021), np.float64(0.02)]),
 ('llama3-70b simp_nl_CandR',
  ['no pair', 0.533, 0.443, 0.448],
  ['no pair', np.float64(0.097), np.float64(0.022), np.float64(0.022)]),
 ('gpt-4o  simp simp_nl_C',
  ['no pair', 0.523, 0.415, 0.421],
  ['no pair', np.float64(0.091), np.float64(0.022), np.float64(0.021)]),
 ('gpt-4o  simp_en_C',
  ['no pair', 0.523, 0.422, 0.428],
  ['no pair', np.float64(0.087), np.float64(0.021), np.float64(0.02)]),
 ('llama3-70b ori_en_C',
  ['no pair', 0.435, 0.293, 0.301],
  ['no pair', np.float64(0.08), np.float64(0.023), np.float64(0.022)]),
 ('llama3-70b ori_nl_CandR',
  ['no pair', 0.609, 0.428, 0.438],
  ['no pair', np.float64(0.077), np.float64(0.023), np.float64(0.022)]),
 ('llama3-70b simp_en_CandR',
  ['no pair', 0.599, 0.49, 0.496],
  ['no pair', np.float64(0.082), np.float64(0.021), np.float64(0.02)]),
 ('gpt-4o  ori_nl_CandR',
  ['no pair', 0.639, 0.50

In [54]:
# For Table 2 - get the 1254 samples that have the same annotations from experts and crowds and calculate the average pairwise IAA among LLM and experts/crowds

import pandas as pd
import os

input_dir = "./gpt and llama"

results = []

def calculate_pairwise_agreement_selected_samples(data_df, annotations_asr_original):

    with open('./batches_annotators.json', 'r') as infile:
        annotators_original = json.load(infile)

    annotations_asr = load_all_annotations_of_all_annotators(annotations_asr_original, data_df)
    group_annotators = get_grouped_annotators(annotators_original, annotations_asr)
    annotations_asr, group_annotators = add_llm_scores(data_df, annotations_asr, group_annotators)
    # converting df to a list with all responses
    triples_results = annotations_asr.values.tolist()
    create_pairwise_alpha_csv(group_annotators, triples_results)
    mean_alpha = calculate_mean_alpha_per_annotator(annotations_asr)
    return mean_alpha



for filename in os.listdir(input_dir):
    if filename.endswith(".csv"):
        model_name = os.path.splitext(filename)[0]
        if model_name != 'model_confusion_matrix_summary':
            filepath = os.path.join(input_dir, filename)
            # load the dataset
            data = pd.read_csv(filepath)

            # calculate the accuracy of the model on the samples where the percentage of lable 1 or 2  is 1
            # get the extract_ids of data_per_1 and data_per_0
            consensus_data = pd.read_csv("./ready_annotations_1254.csv")
            # get the extract_ids of data_per_1 and data_per_0
            extract_ids_consensus = consensus_data['extract_id'].unique()
            # filter the data where the extract_id is in data_per_1 or data_per_0
            filtered_data = data[data['extract_id'].isin(extract_ids_consensus)]

            with open("./ready_annotations.csv", 'r') as f:
                annotations_asr_original = pd.read_csv(f)
                filtered_annotations_asr_original = annotations_asr_original[annotations_asr_original['extract_id'].isin(extract_ids_consensus)]
            # calculate the IAA
            alphas = calculate_pairwise_agreement_selected_samples(filtered_data, filtered_annotations_asr_original)
            
            # get the alpha for 'llm'
            alpha_llm = alphas[alphas['anonymised_participant_id'] == 'llm']['mean_alpha'].values[0]
            alpha_moe = alphas[alphas['anonymised_participant_id'] == 'llm']['moe'].values[0]
            
            results.append((model_name, alpha_llm, alpha_moe))

results

/Users/yahui/Contentious-term-recognition-with-LLM/contentious/lib/python3.9/site-packages/numpy/_core/_methods.py:218: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/yahui/Contentious-term-recognition-with-LLM/contentious/lib/python3.9/site-packages/numpy/_core/_methods.py:210: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/Users/yahui/Contentious-term-recognition-with-LLM/contentious/lib/python3.9/site-packages/numpy/_core/_methods.py:218: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/yahui/Contentious-term-recognition-with-LLM/contentious/lib/python3.9/site-packages/numpy/_core/_methods.py:210: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/Users/yahui/Contentious-term-recognition-with-LLM/contentious/lib/python3.9/site-packages/numpy/_core/_meth

[('gpt-4o  ori_en_CandR',
  ['no pair', 0.973, 0.743, 0.756],
  ['no pair', np.float64(0.023), np.float64(0.032), np.float64(0.031)]),
 ('llama3-70b simp_nl_CandR',
  ['no pair', 0.567, 0.696, 0.689],
  ['no pair', np.float64(0.227), np.float64(0.038), np.float64(0.038)]),
 ('gpt-4o  simp simp_nl_C',
  ['no pair', 0.697, 0.493, 0.505],
  ['no pair', np.float64(0.077), np.float64(0.033), np.float64(0.031)]),
 ('gpt-4o  simp_en_C',
  ['no pair', 0.763, 0.494, 0.509],
  ['no pair', np.float64(0.064), np.float64(0.032), np.float64(0.031)]),
 ('llama3-70b ori_en_C',
  ['no pair', 0.594, 0.297, 0.313],
  ['no pair', np.float64(0.048), np.float64(0.03), np.float64(0.029)]),
 ('llama3-70b ori_nl_CandR',
  ['no pair', 0.827, 0.514, 0.531],
  ['no pair', np.float64(0.078), np.float64(0.035), np.float64(0.034)]),
 ('llama3-70b simp_en_CandR',
  ['no pair', 0.786, 0.735, 0.738],
  ['no pair', np.float64(0.117), np.float64(0.032), np.float64(0.031)]),
 ('gpt-4o  ori_nl_CandR',
  ['no pair', 1.0, 0.